# Final Project

Author: Evan Whitfield

Course: ST 554

Purpose: Final Project

## Part 1 - Fitting Your Model

First, we need to import all the necessary modules for our task.

In [24]:
import pandas as pd
import numpy as np

from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, Binarizer, OneHotEncoder, VectorAssembler, PCA
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col

We need to read in our data for our machine learning process. This will serve as our training set for our model building process.

In [48]:
power_ml_data = pd.read_csv('power_ml_data.csv')

Time to start the spark session and turn the data into a spark dataframe.

In [26]:
spark_sess = SparkSession.builder.getOrCreate()

spark_df = spark_sess.createDataFrame(power_ml_data)

In [49]:
spark_df.show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
|      5.853|    76.9|     0.081|       

In [50]:
spark_df.printSchema()

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



We want the `Hour` variable to be a Double type instead of a Long type. 

In [29]:
cast_hour_double = SQLTransformer(statement = 
    """
    SELECT *, CAST(Hour AS DOUBLE) AS Hour_double
    FROM __THIS__
    """
)

Next, we want to change our hour variable into a binary for either day or night time using a threshold of 6.5.

In [30]:
binary_day_night = Binarizer(threshold=6.5, inputCol="Hour_double", outputCol="day_night")

We want to One Hoe Encode the month variable.

In [31]:
one_hot = OneHotEncoder(inputCols=["Month"],outputCols=["Month_ohe"],dropLast=False)

Time to assemble the different weather related variables into a single column.

In [32]:
weather_assembled = VectorAssembler(
    inputCols=["Temperature","Humidity","Wind_Speed","General_Diffuse_Flows","Diffuse_Flows"],
    outputCol="weather",
    handleInvalid="skip"
)

The next transform we want to do is a PCA with k = 2 and store the results in a new column.

In [51]:
pca = PCA(
    k=2,
    inputCol="weather",
    outputCol="pca_features"
)

We need to set up our label and features column to get ready for the Linear Regression.

In [34]:
label_maker = SQLTransformer(statement="""
    SELECT *, Power_Zone_3 AS label
    FROM __THIS__
    """
)

In [35]:
features_assembler = VectorAssembler(
    inputCols=["pca_features","day_night","Power_Zone_1","Power_Zone_2","Month_ohe"],
    outputCol="features",
    handleInvalid="skip"
)

In [36]:
lr = LinearRegression()

Next, we want to set up our grid of possible parameters to test to check to see what will produce the lowest RMSE.

In [37]:
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

Time to put all of our different transforms into a pipeline, ending with the Linear Regression.

In [38]:
pipeline = Pipeline(stages = [
    cast_hour_double, 
    binary_day_night, 
    one_hot, 
    weather_assembled,
    pca,
    label_maker,
    features_assembler,
    lr
])

The pipeline will be used as our estimator with 5 folds. We want to determine which set of parameters gives us the lowest RMSE.

In [39]:
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=RegressionEvaluator(metricName='rmse'),
    numFolds=5,
    parallelism = 8
)

In [40]:
cvModel = cv.fit(spark_df)

26/04/29 22:14:39 WARN BlockManager: Block rdd_46311_15 already exists on this machine; not re-adding it
26/04/29 22:14:42 WARN Instrumentation: [47dba27a] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 22:14:42 WARN Instrumentation: [c022eee5] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 22:14:42 WARN Instrumentation: [1e276a36] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 22:14:42 WARN Instrumentation: [40b199cb] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 22:14:42 WARN Instrumentation: [8cf4ad97] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 22:14:42 WARN Instrumentation: [4e652b2b] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 22:14:43 WARN Instrumentation: [c022eee5] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton sol

We need to do use a couple of attributes to reside to produce the best parameters for reducing the RMSE.

In [41]:
best_pipeline_model = cvModel.bestModel
best_lr_model = best_pipeline_model.stages[-1] 

print("Best regParam:", best_lr_model._java_obj.getRegParam())
print("Best elasticNetParam:", best_lr_model._java_obj.getElasticNetParam())

Best regParam: 0.5
Best elasticNetParam: 1.0


In [42]:
rmse_values = cvModel.avgMetrics
best_rmse = min(rmse_values)

print("Best CV RMSE:", best_rmse)

Best CV RMSE: 2148.0704011494704


In [44]:
preds = cvModel.transform(spark_df).select("label", "features", "prediction")

In [45]:
test_error = RegressionEvaluator().evaluate(cvModel.transform(spark_df))
print(test_error)

2147.0988830702504


In [46]:
preds_with_resids = preds.withColumn("residual",
    col("label") - col("prediction")
)

In [53]:
preds_with_resids.show(20)

+-----------+--------------------+------------------+------------------+
|      label|            features|        prediction|          residual|
+-----------+--------------------+------------------+------------------+
|20240.96386|(18,[0,1,3,4,6],[...| 20877.12110429421|-636.1572442942088|
|20131.08434|(18,[0,1,3,4,6],[...|  18655.6110409688|1475.4732990312004|
|19668.43373|(18,[0,1,3,4,6],[...|18200.241325955576|1468.1924040444246|
|18899.27711|(18,[0,1,3,4,6],[...|17586.380086048153|1312.8970239518458|
|18442.40964|(18,[0,1,3,4,6],[...|16993.176745584413|1449.2328944155888|
|18130.12048|(18,[0,1,3,4,6],[...|16513.718382503597|1616.4020974964042|
|17945.06024|(18,[0,1,3,4,6],[...|16089.429750681884| 1855.630489318115|
|17459.27711|(18,[0,1,3,4,6],[...|15718.986378992151| 1740.290731007848|
|17025.54217|(18,[0,1,3,4,6],[...| 15267.48585326926| 1758.056316730741|
|16794.21687|(18,[0,1,3,4,6],[...|14934.880435957024|1859.3364340429762|
|16638.07229|(18,[0,1,3,4,6],[...|14649.16499834266

## Part 2 - Streaming Part